# Rosati 2022 — Master Table Construction (Original Data from Elisa Rosati)

**Source**: Direct data transfer from Elisa Rosati (corresponding author), received via WeTransfer, containing:
- TCRαβ bulk repertoires (RDS format) for three groups: Crohn's Disease (CD), Healthy controls, and Ulcerative Colitis (UC, labelled "colitis" in source files)
- HIBAG-imputed HLA genotypes (SNP-array based) for CD and Healthy groups only

**Study**: Rosati et al. 2022, *A novel unconventional T cell population enriched in Crohn's disease*, Gut, doi:10.1136/gutjnl-2021-325373. ENA accession PRJEB50045.

**Note on nomenclature**: "CD" in this dataset refers to Crohn's Disease, not Celiac Disease. The third clinical group ("colitis" in source filenames) corresponds to Ulcerative Colitis (UC) patients in the original publication.

**Scope of this notebook**: Raw data unpacking and master table construction only. No quality filtering is applied here — this table represents the original data as received. QC filtering, blind HLA inference (HLAGuessr/THNet), and validation against this ground truth are performed in the companion notebook.

## 0. Setup

In [ ]:
import pandas as pd
import json
import os

BASE_WETRANSFER = '/home/immunologylab/Downloads/wetransfer_cait-hla-and-tcr-data_2026-06-12_0708/'
DATA            = '/home/immunologylab/bioinformatics/hla-tcr-project/data/'
TCR_DIR         = DATA + 'rosati_tcr_raw/'
CDR_DICT_DIR    = '/home/immunologylab/bioinformatics/TRA_TRB_pairingDB/Oleg_DS_pairedCDR3/'

os.makedirs(TCR_DIR, exist_ok=True)

## 1. Unpack TCR repertoires (RDS → TSV)

Source RDS files contain one list per chain/group combination, with one data.frame per patient (named e.g. `1.CD.3.Blood.bulk`). Each per-patient TSV is written with an explicit chain suffix (`_TRA.tsv` / `_TRB.tsv`) to avoid filename collisions between chains for the same patient.

This step is performed in R (see `unpack_rosati_rds.R` in this repository) and produces 480 TSV files: 240 TRA + 240 TRB, covering 106 CD + 98 Healthy + 36 UC patients.

In [ ]:
# R script (run separately, see unpack_rosati_rds.R):
#
# files <- list(
#   TRA_Crohn="220716_TRA_Crohn.rds", TRA_Healthy="220716_TRA_healthy.rds", TRA_UC="220716_TRA_colitis.rds",
#   TRB_Crohn="220716_TRB_Crohn.rds", TRB_Healthy="220716_TRB_healthy.rds", TRB_UC="220716_TRB_colitis.rds"
# )
# for (nm in names(files)) {
#   chain <- ifelse(grepl("TRA", nm), "TRA", "TRB")
#   data  <- readRDS(paste0(base_path, files[[nm]]))
#   for (i in seq_along(data)) {
#     write.table(data[[i]], file=paste0(out_dir, names(data)[i], "_", chain, ".tsv"),
#                 sep="\t", quote=FALSE, row.names=FALSE, col.names=TRUE)
#   }
# }

tra_files = [f for f in os.listdir(TCR_DIR) if f.endswith('_TRA.tsv')]
trb_files = [f for f in os.listdir(TCR_DIR) if f.endswith('_TRB.tsv')]
print(f'TRA files: {len(tra_files)} | TRB files: {len(trb_files)}')

## 2. Unpack HLA ground truth (HIBAG dosage format)

HLA files are in HIBAG dosage format: rows are alleles (`imputed_HLA_X*YY:ZZ`), columns are patient IDs, values are genotype dosage (0 = absent, 1 = heterozygous, 2 = homozygous). Only CD and Healthy groups have HLA data — Elisa Rosati confirmed UC/colitis HLA was not part of the shared subset.

Dosage is converted to long format (`patient_id | allele | dosage`) and a binary `true_carrier` flag (`dosage >= 1`) is derived.

In [ ]:
hla_crohn   = pd.read_csv(BASE_WETRANSFER + '220717_inputedHLA_Crohn.tsv',   sep='\t', check_names=False)
hla_healthy = pd.read_csv(BASE_WETRANSFER + '220717_inputedHLA_Healthy.tsv', sep='\t', check_names=False)

def to_long(hla_df, group):
    meta_cols   = ['chr','id','pos','REF','ALT','INFO','TYPE','source']
    sample_cols = [c for c in hla_df.columns if c not in meta_cols]
    alleles     = hla_df['id'].str.replace('imputed_HLA_', '', regex=False)
    rows = []
    for pid in sample_cols:
        rows.append(pd.DataFrame({'patient_id': pid, 'allele': alleles,
                                   'dosage': hla_df[pid], 'group': group}))
    return pd.concat(rows, ignore_index=True)

long_crohn   = to_long(hla_crohn,   'CD')
long_healthy = to_long(hla_healthy, 'Healthy')
hla_real     = pd.concat([long_crohn, long_healthy], ignore_index=True)
hla_real['true_carrier'] = hla_real['dosage'] >= 1

hla_real.to_csv(DATA + 'rosati_real_hla_long.tsv', sep='\t', index=False)
print(f'Patients: {hla_real["patient_id"].nunique()} | Alleles: {hla_real["allele"].nunique()}')

## 3. Annotate CDR1/CDR2 from V-gene

Source TSVs contain CDR3 (`aaSeqCDR3`) and V/J gene calls (`bestVHit`, `bestJHit`) but no CDR1/CDR2 — these are germline-encoded and depend only on the V-gene allele, not on recombination. CDR1/CDR2 are looked up from pre-built per-V-gene dictionaries (same dictionaries used elsewhere in this project for TRA/TRB CDR annotation).

In [ ]:
with open(CDR_DICT_DIR + 'v_cdr_dict.json') as f:
    v_cdr_dict = json.load(f)  # TRBV -> [CDR1, CDR2]
with open(CDR_DICT_DIR + 'a_cdr_dict.json') as f:
    a_cdr_dict = json.load(f)  # TRAV -> [CDR1, CDR2]

def norm_vgene(v):
    return str(v).split('*')[0].strip() if pd.notna(v) else None

def get_cdr1(v, chain):
    d = a_cdr_dict if chain == 'TRA' else v_cdr_dict
    return d.get(norm_vgene(v), [None, None])[0]

def get_cdr2(v, chain):
    d = a_cdr_dict if chain == 'TRA' else v_cdr_dict
    return d.get(norm_vgene(v), [None, None])[1]

## 4. Build TCR table — CD + Healthy only

UC/colitis patients are excluded from this master table since no HLA ground truth is available for that group. All TRA + TRB files for CD and Healthy are loaded, V/J genes normalised (IMGT allele suffix removed), and CDR1/CDR2 annotated.

In [ ]:
frames = []
target_files = [f for f in sorted(os.listdir(TCR_DIR))
                if (f.endswith('_TRA.tsv') or f.endswith('_TRB.tsv'))
                and '.UC.' not in f]

for f in target_files:
    chain = 'TRA' if f.endswith('_TRA.tsv') else 'TRB'
    pid   = f.replace('_TRA.tsv', '').replace('_TRB.tsv', '')
    group = 'CD' if '.CD.' in pid else 'Healthy'

    df = pd.read_csv(TCR_DIR + f, sep='\t')
    df['patient_id'] = pid
    df['chain']      = chain
    df['group']      = group
    df['v_gene']     = df['bestVHit'].apply(norm_vgene)
    df['j_gene']     = df['bestJHit'].apply(lambda x: norm_vgene(x))
    df['CDR1']       = df.apply(lambda r: get_cdr1(r['bestVHit'], chain), axis=1)
    df['CDR2']       = df.apply(lambda r: get_cdr2(r['bestVHit'], chain), axis=1)
    frames.append(df)

tcr_all = pd.concat(frames, ignore_index=True)
print(f'Clones: {len(tcr_all):,} | Patients: {tcr_all["patient_id"].nunique()} | '
      f'Chains: {tcr_all["chain"].value_counts().to_dict()}')

## 5. Join TCR with HLA ground truth and save

HLA long-format table is pivoted to wide (one column per allele) and left-joined onto the TCR table by `patient_id`. Patients present in TCR data but absent from the HLA file (a subset Elisa Rosati did not include) are saved separately, without HLA columns.

In [ ]:
hla_wide = hla_real.pivot_table(index='patient_id', columns='allele',
                                  values='dosage', aggfunc='first').reset_index()
hla_wide.columns.name = None

tcr_final = tcr_all[['patient_id', 'group', 'chain', 'v_gene', 'j_gene',
                      'CDR1', 'CDR2', 'aaSeqCDR3', 'cloneCount', 'cloneFraction']]\
            .rename(columns={'aaSeqCDR3': 'CDR3'})

master = tcr_final.merge(hla_wide, on='patient_id', how='left')
master['has_hla'] = master['patient_id'].isin(hla_wide['patient_id'].values)

master_with_hla    = master[master['has_hla']].drop(columns=['has_hla'])
master_without_hla = master[~master['has_hla']].drop(
    columns=['has_hla'] + [c for c in master.columns if '*' in str(c)])

master_with_hla.to_csv(DATA + 'rosati_master_tcr_hla.tsv.gz',
                        sep='\t', index=False, compression='gzip')
master_without_hla.to_csv(DATA + 'rosati_master_tcr_nohla.tsv.gz',
                           sep='\t', index=False, compression='gzip')

print(f'Patients with HLA:    {master_with_hla["patient_id"].nunique()} '
      f'({len(master_with_hla):,} clones)')
print(f'Patients without HLA: {master_without_hla["patient_id"].nunique()} '
      f'({len(master_without_hla):,} clones)')

## 6. Summary

- **204 patients** unpacked from Elisa Rosati's source RDS files: 106 CD (Crohn's Disease), 98 Healthy, 36 UC (no HLA available, excluded from master table).
- **192 patients (98 CD + 94 Healthy)** have both TCRαβ and HIBAG-imputed HLA ground truth — these form `rosati_master_tcr_hla.tsv.gz`.
- **12 patients** have TCR but no HLA (subset not shared by Elisa Rosati) — saved separately in `rosati_master_tcr_nohla.tsv.gz`.
- This table contains **no quality filtering** — it reflects the original MiXCR clonotype calls as provided. QC (productive/canonical CDR3, minimum clone depth) and downstream HLA inference validation are performed in the companion notebook.
- **Nomenclature correction**: "CD" = Crohn's Disease (not Celiac Disease) per the original publication (Gut, 2022, doi:10.1136/gutjnl-2021-325373).